# Day 089 Solution — Mini Portfolio Data Store

In [ ]:
import pandas as pd, math, sqlite3, tempfile, os

OHLCV_COLS = ["Open", "High", "Low", "Close", "Volume"]

def _synthetic(ticker="TEST", period="1y", interval="1d", n=50):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
def validate_ohlcv(df):
    if not isinstance(df, pd.DataFrame): return False, "not a DataFrame"
    missing = [c for c in OHLCV_COLS if c not in df.columns]
    if missing: return False, "missing columns: " + ", ".join(missing)
    if len(df) == 0: return False, "DataFrame is empty"
    valid = df.dropna(subset=["High", "Low"])
    if len(valid) > 0 and (valid["High"] < valid["Low"]).any():
        return False, "High < Low detected"
    return True, ""
def normalize_ohlcv(df):
    df = df.copy()
    if not isinstance(df.index, pd.DatetimeIndex): df.index = pd.to_datetime(df.index)
    if df.index.tz is not None: df.index = df.index.tz_localize(None)
    return df[[c for c in OHLCV_COLS if c in df.columns]]
def fetch_ohlcv(ticker, period="1y", interval="1d", fetch_fn=None):
    if fetch_fn is not None: return fetch_fn(ticker, period, interval)
    import yfinance as yf
    df = yf.download(ticker, period=period, interval=interval, progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    return df
def store_ohlcv(df, ticker, db_path=":memory:"):
    ok, reason = validate_ohlcv(df)
    if not ok: raise ValueError("Invalid OHLCV: " + reason)
    df = normalize_ohlcv(df)
    con = sqlite3.connect(db_path)
    try:
        con.execute("CREATE TABLE IF NOT EXISTS ohlcv (ticker TEXT NOT NULL, date TEXT NOT NULL, open REAL, high REAL, low REAL, close REAL, volume REAL, PRIMARY KEY (ticker, date))")
        rows = []
        for date, row in df.iterrows():
            d = str(date.date()) if hasattr(date, "date") else str(date)
            rows.append((ticker, d, float(row["Open"]), float(row["High"]),
                         float(row["Low"]), float(row["Close"]), float(row["Volume"])))
        con.executemany("INSERT OR REPLACE INTO ohlcv VALUES (?,?,?,?,?,?,?)", rows)
        con.commit(); return len(rows)
    finally: con.close()

def load_ohlcv(ticker, db_path=":memory:"):
    con = sqlite3.connect(db_path)
    try:
        try:
            rows = con.execute(
                "SELECT date, open, high, low, close, volume FROM ohlcv "
                "WHERE ticker=? ORDER BY date", (ticker,)).fetchall()
        except sqlite3.OperationalError:
            return pd.DataFrame(columns=OHLCV_COLS)
        if not rows: return pd.DataFrame(columns=OHLCV_COLS)
        df = pd.DataFrame(rows, columns=["date", "Open", "High", "Low", "Close", "Volume"])
        df.index = pd.to_datetime(df["date"]); df.index.name = None
        return df.drop(columns=["date"])
    finally: con.close()
class MarketDataStore:
    def __init__(self, db_path=":memory:", fetch_fn=None):
        self._db = db_path; self._fetch_fn = fetch_fn; self._tickers = set()
    def fetch(self, ticker, period="1y", interval="1d"):
        return fetch_ohlcv(ticker, period=period, interval=interval, fetch_fn=self._fetch_fn)
    def update(self, ticker, period="1y", interval="1d"):
        df = self.fetch(ticker, period, interval)
        n = store_ohlcv(df, ticker, self._db); self._tickers.add(ticker); return n
    def load(self, ticker): return load_ohlcv(ticker, self._db)
    def tickers(self): return sorted(self._tickers)


In [ ]:
def _synthetic_ohlcv(ticker="TEST", period="1y", interval="1d"):
    import math
    n = 252
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / 252)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })


In [ ]:
import tempfile, os

_db = tempfile.NamedTemporaryFile(suffix=".db", delete=False)
DB_PATH = _db.name
_db.close()

store = MarketDataStore(db_path=DB_PATH, fetch_fn=_synthetic_ohlcv)

for ticker in ["AAPL", "MSFT", "GOOG"]:
    n = store.update(ticker)
    assert n == 252, f"expected 252 rows for {ticker}, got {n}"

assert store.tickers() == ["AAPL", "GOOG", "MSFT"], f"tickers={store.tickers()}"


In [ ]:
for ticker in store.tickers():
    df = store.load(ticker)
    ok, reason = validate_ohlcv(df)
    assert ok, f"{ticker}: {reason}"
    assert isinstance(df.index, pd.DatetimeIndex)
    assert len(df) == 252
    print(f"{ticker}: {len(df)} rows | close mean={df['Close'].mean():.2f}")


In [ ]:
aapl = store.load("AAPL")
csv_path = DB_PATH.replace(".db", "_AAPL.csv")
aapl.to_csv(csv_path)
assert os.path.exists(csv_path)
print(f"\nExported {len(aapl)} rows to {csv_path}")


In [ ]:
os.unlink(DB_PATH)
os.unlink(csv_path)
print("Solution smoke-test passed.")
